In [1]:
import pyroomacoustics as pra

import os
from tqdm import tqdm

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torch.autograd import profiler
import torchaudio
from torchaudio.transforms import Resample
from torchmetrics.audio import SpeechReverberationModulationEnergyRatio, ShortTimeObjectiveIntelligibility


from einops import rearrange

from src.dataset import SignalDataset, TRUNetDataset
from src.loss import loss_tot, loss_MR, loss_MR_w
from models.fspen import * # FullSubPathExtension, FullSubPathExtension_ver2, FullSubPathExtension_abs_pha, FullSubPathExtension_abs_pha_mapping, FullSubPathExtension_ver3

from IPython.display import Audio

from src.utils import model_eval, model_eval_fspen2x_ver3, model_eval_old

import matplotlib.pyplot as plt

In [2]:
CHKP_DIR = "checkpoints"

np.set_printoptions(precision=3)
torch.set_printoptions(precision=3)

In [3]:
from src.fspen_configs import *

configs = TrainConfig_48kHz_enc_ext()
# print(sum(configs.bands_num_in_groups), configs.dual_path_extension["num_modules"])
fspen = FullSubPathExtension_ver2_abs_pha_TRA(configs=configs)# .to(DEVICE)

state_d = torch.load(os.path.join(CHKP_DIR, "fspen_chkp", "TrainConfig_48kHz_enc_ext_tra#0.pt"), map_location="cpu",  weights_only=False)

64


In [4]:
# N_FFTS = 512
# HOP_LENGTH = 256
# HID_SIZE = 32
# SR = 16_000

N_FFTS = configs.n_fft
HOP_LENGTH = configs.hop_length
HID_SIZE = 64
SR = configs.sample_rate

DEVICE = "cpu" # torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"It's {DEVICE} time!!!")

It's cpu time!!!


In [5]:
fspen.load_state_dict(state_d["model_state_dict"])
fspen.eval()

FullSubPathExtension_ver2_abs_pha_TRA(
  (full_band_encoder): TRAFullBandEncoder(
    (full_band_encoder): ModuleList(
      (0): FullBandEncoderBlock(
        (conv): Conv1d(2, 4, kernel_size=(6,), stride=(2,), padding=(2,))
        (norm): BatchNorm1d(4, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (activate): ELU(alpha=1.0)
      )
      (1): FullBandEncoderBlock(
        (conv): Conv1d(4, 16, kernel_size=(8,), stride=(2,), padding=(3,))
        (norm): BatchNorm1d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (activate): ELU(alpha=1.0)
      )
      (2): FullBandEncoderBlock(
        (conv): Conv1d(16, 32, kernel_size=(6,), stride=(2,), padding=(2,))
        (norm): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (activate): ELU(alpha=1.0)
      )
    )
    (global_features): Conv1d(32, 32, kernel_size=(1,), stride=(1,))
    (tra_conv1): TRAEncoderBlock(
      (conv1): Conv1d(16, 48, ke

In [6]:
def vorbis_window(winlen, device="cuda"):
    sq = torch.sin(torch.pi/2*(torch.sin(torch.pi/winlen*(torch.arange(winlen)-0.5))**2)).float()
    return sq

In [7]:
AUDIO_PATH = "input/test_act.wav" # "data/real/ade9ba35-e411-4a57-af8e-7fb9cc7e281b_24-04-10_19-43-24_u587277047485_sCAMERA_v373_audioProcessor_input_0_AUDIO_ONLY.wav" # "input/test_act.wav"

In [10]:
signal, signal_sr = torchaudio.load(AUDIO_PATH)

input_signal, _ = SignalDataset.normalize_audio(signal)
if signal_sr != SR:
    resampler = Resample(signal_sr, SR)
    input_signal = resampler(input_signal)
# input_signal = input_signal[..., SR * 5:SR * 20]

In [11]:
Audio(input_signal, rate=SR)

In [12]:
window = vorbis_window(N_FFTS)

spec = torch.stft(
            input_signal,
            n_fft=N_FFTS,
            hop_length=HOP_LENGTH,
            # onesided=True,
            win_length=N_FFTS,
            window=window,
            return_complex=True,
            normalized=True,
            center=True
        )

output, _ = model_eval(fspen, spec, configs, DEVICE, hid_size=64)

out_wave = torch.istft(output, n_fft=N_FFTS, hop_length=HOP_LENGTH, win_length=N_FFTS,
                       window=window,
                       # onesided=True,
                       return_complex=False,
                       normalized=True,
                       center=True)

# out_wave = out_wave / (out_wave.abs().max() / input_signal.abs().max())

# out_wave = torch.istft(spec, n_fft=N_FFTS, hop_length=HOP_LENGTH, win_length=N_FFTS,
#                        window=window,
#                        # onesided=True,
#                        return_complex=False,
#                        normalized=True,
#                        center=False)

out_wave = out_wave.reshape(-1)

In [13]:
Audio(out_wave.detach(), rate=SR)

In [14]:
from scipy.io.wavfile import write

write(AUDIO_PATH[:-4] + "_enc_ext_tra.wav", SR, out_wave.cpu().detach().numpy())
# write(AUDIO_PATH.split("/")[-1][:-4] + "_baseline.wav", SR, out_wave.cpu().detach().numpy())

In [15]:
import yaml

from NISQA_s.src.core.model_torch import model_init
from NISQA_s.src.utils.process_utils import process

NISQA_PATH = "NISQA_s/config/nisqa_s.yaml"

with open(NISQA_PATH, 'r') as stream:
    nisqa_args = yaml.safe_load(stream)
nisqa_args["ms_n_fft"] = 512
nisqa_args["hop_length"] = 256
nisqa_args["ms_win_length"] = 512
nisqa_args["ckp"] = nisqa_args["ckp"][3:]

nisqa, h0_nisqa, c0_nisqa = model_init(nisqa_args)

/home/zakhar/miniconda3/envs/ems_dereverb/lib/python3.10/site-packages/torch/nn/modules/rnn.py:83: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=1 and num_layers=1
  warnings.warn("dropout option adds dropout after all but last "


In [16]:
from torch_stoi import NegSTOILoss
from torchmetrics.audio.pesq import PerceptualEvaluationSpeechQuality

srmr = SpeechReverberationModulationEnergyRatio(fs=16_000, norm=False)
stoi = NegSTOILoss(SR, use_vad=False, do_resample=False).to(DEVICE)
pesq = PerceptualEvaluationSpeechQuality(fs=16_000, mode="wb").to(DEVICE)

In [17]:
target, signal_sr = torchaudio.load("input/gt_act.wav")

# if signal_sr != SR:
#     resampler = Resample(signal_sr, SR)
#     target = resampler(target)

# target, _ = SignalDataset.normalize_audio(target)

In [18]:
# min_l = min(out_wave.shape[-1], input_signal.shape[-1])
# input_signal = signal
min_l = min(out_wave.shape[-1], input_signal.shape[-1], target.shape[-1])
nisqa_score_in, _, _ = process(input_signal[..., :min_l].unsqueeze(0), SR, nisqa, h0_nisqa, c0_nisqa, nisqa_args)
nisqa_score_out, _, _ = process(out_wave[..., :min_l].unsqueeze(0), SR, nisqa, h0_nisqa, c0_nisqa, nisqa_args)
nisqa_score_target, _, _ = process(target[..., :min_l].unsqueeze(0), SR, nisqa, h0_nisqa, c0_nisqa, nisqa_args)
print("NISQA in:", nisqa_score_in)
print("NISQA out:", nisqa_score_out)
print("NISQA target:", nisqa_score_target)

stoi_score_in = -stoi(input_signal[..., :min_l], target[..., :min_l])
stoi_score_out = -stoi(out_wave[..., :min_l].unsqueeze(0), target[..., :min_l])
stoi_score_target = -stoi(target[..., :min_l], target[..., :min_l])
print(f"STOI in:", stoi_score_in.item())
print(f"STOI out:", stoi_score_out.item())
print(f"STOI out:", stoi_score_target.item())

resampler = Resample(SR, 16_000)
input_signal = resampler(input_signal.cpu())
out_wave = resampler(out_wave.cpu())
target = resampler(target)
# input_signal = resampler(input_signal.cpu()).cuda()
# min_l = min(out_wave.shape[-1], target.shape[-1])
min_l = min(out_wave.shape[-1], input_signal.shape[-1], target.shape[-1])
srmr_score_in = srmr(input_signal[..., :min_l].detach())
srmr_score_out = srmr(out_wave[..., :min_l].detach())
srmr_score_target = srmr(target[..., :min_l])
print(f"SRMR in: {srmr_score_in:.2f}")
print(f"SRMR out: {srmr_score_out:.2f}")
print(f"SRMR target: {srmr_score_target:.2f}")

# min_l = min(out_wave.shape[-1], input_signal.shape[-1], target.shape[-1])

# print(out_wave.shape, target.shape, input_signal.shape, min_l)
pesq_score_in = pesq(input_signal[0, :min_l], target[0, :min_l])
pesq_score_out = pesq(out_wave[..., :min_l], target[0, :min_l])
pesq_score_targer = pesq(target[0, :min_l], target[0, :min_l])

print(f"PESQ in: {pesq_score_in:.2f}")
print(f"PESQ out: {pesq_score_out:.2f}")
print(f"PESQ target: {pesq_score_targer:.2f}")

NISQA in: tensor([[2.020, 2.311, 3.148, 2.733, 2.672]])
NISQA out: tensor([[3.871, 4.117, 3.825, 3.948, 4.023]])
NISQA target: tensor([[1.309, 1.815, 3.036, 2.036, 2.451]])
STOI in: 0.8206276297569275
STOI out: 0.8700284957885742
STOI out: 1.0
SRMR in: 7.93
SRMR out: 9.71
SRMR target: 12.23
PESQ in: 1.61
PESQ out: 2.44
PESQ target: 4.64


NISQA in: tensor([[2.020, 2.311, 3.148, 2.733, 2.672]])
NISQA out: tensor([[4.200, 4.294, 4.143, 4.104, 4.164]])
NISQA target: tensor([[1.309, 1.815, 3.036, 2.036, 2.451]])
STOI in: 0.8206276297569275
STOI out: 0.8747361898422241
STOI out: 1.0
SRMR in: 7.93
SRMR out: 11.98
SRMR target: 12.23
PESQ in: 1.61
PESQ out: 2.68
PESQ target: 4.64